# Stop Times Checks

Checks trip duplication, stop_sequence regularity, and route-following behavior in stop_times.

In [ ]:
import csv
import importlib
import os
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from statistics import mean, stdev
import sys
from typing import Dict, List, Set, Tuple

_current = Path.cwd().resolve()
for _candidate in [_current, *_current.parents]:
    if (_candidate / "data_validation" / "gtfs_utils.py").exists():
        _project_root = _candidate
        break
else:
    raise FileNotFoundError("data_validation/gtfs_utils.py not found.")

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

import scripts.basics as basics
import data_validation.gtfs_utils as gtfs_utils
gtfs_utils = importlib.reload(gtfs_utils)
from data_validation.gtfs_utils import (
    DUPLICATED_TRIPS_BASE,
    STOP_SEQUENCE_BASE,
    DOORS_BASE,
    DOORS_FILE,
    STOP_TIMES_SUBWAY_FILE,
    STOP_TIMES_CLEANED_FILE,
    STOP_TIMES_SEQUENCE_FILE,
    STOP_TIMES_FILE,
    STOPS_FILE,
    TRIPS_SUBWAY_FILE,
    TRIPS_FILE,
    TRIP_IDS_TO_ELIMINATE_FILE,
    WRONG_STOP_SEQUENCES_FILE,
    check_missing_files,
    print_file_disclaimer,
    load_stop_names,
    load_trip_ids,
    load_trip_to_line,
    load_nonempty_lines,
    load_trip_sequence_bounds,
    parse_time_to_seconds,
    round_half_up_mean,
    sniff_dialect,
    read_dict_rows,
    build_expected_adjacency,
    collect_trip_stop_ids,
    make_signature,
    check_trip,
)

#### Does stop_sequence increment by one?

Goal: check whether stop_sequence in stop_times_subway.txt increments by one. To do this, we read stop_times once and aggregate the stop_sequence by trip_id for each trip_id in trips_subway.txt. This only depends on `processing/1_subway.py`; trip duplicates don't affect a trip's own sequence continuity, so it doesn't need the deduplication step.

In [ ]:
def main():
    """Check that each trip has stop_sequence values that increment by one."""
    trip_ids = None
    seq_by_trip: Dict[str, Set[int]] = {}
    dict_seq: Dict[str, List[int]] = {}
    max_workers = 0
    violations_total = 0
    check_missing_files([STOP_TIMES_SUBWAY_FILE, TRIPS_SUBWAY_FILE])

    print_file_disclaimer([
        (STOP_TIMES_SUBWAY_FILE, 'stop_times'),
        (TRIPS_SUBWAY_FILE, 'trips'),
    ])

    trip_ids = load_trip_ids(TRIPS_SUBWAY_FILE)

    print(f"Number of trip_id in 'trips': {len(trip_ids)}")

    # Build: trip_id -> set of stop_sequence (no duplicates), in a single pass over stop_times_subway
    seq_by_trip = defaultdict(set)
    for r in read_dict_rows(STOP_TIMES_SUBWAY_FILE):
        tid = r.get('trip_id', '')
        if tid not in trip_ids:
            continue
        seq_str = r.get('stop_sequence', '')
        try:
            seq = int(seq_str)
        except Exception:
            # Ignore non-numeric or empty values
            continue
        seq_by_trip[tid].add(seq)

    # Final map: trip_id -> sorted list of stop_sequence (no duplicates)
    dict_seq = {}
    for trip_id in trip_ids:
        dict_seq[trip_id] = sorted(seq_by_trip.get(trip_id, set()))

    # Parallel validation
    max_workers = min(8, (os.cpu_count() or 4))
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {
            ex.submit(check_trip, trip_id, dict_seq[trip_id]): trip_id for trip_id in trip_ids
        }
        for fut in as_completed(futures):
            msgs = fut.result()
            violations_total += len(msgs)
            for m in msgs:
                print(m)

    if violations_total == 0:
        print(
            "Correct: all trips in 'trips' have stop_sequence in 'stop_times'"
            " that increments by one."
        )
    else:
        print(f"Total violations detected: {violations_total}")

main()

#### Duplicate full trip_id blocs in stop_times

In [ ]:
def build_trip_stop_events(stop_times_file, trip_ids):
    """Read stop_times and group ordered stop-event tuples by trip_id.

    args:
        stop_times_file: Path to the stop_times CSV file.
        trip_ids: Set of trip_id values to include.

    returns:
        Tuple of (mapping from trip_id to its (seq, arrival, departure, stop_id) tuples,
        total rows read from the file).
    """
    trips_rows: Dict[str, List[Tuple[int, str, str, str]]] = defaultdict(list)
    total_rows = 0
    for row in read_dict_rows(stop_times_file):
        total_rows += 1
        trip_id = row.get("trip_id", "")
        if trip_id not in trip_ids:
            continue
        sequence_text = row.get("stop_sequence", "")
        arrival_time = row.get("arrival_time", "")
        departure_time = row.get("departure_time", "")
        stop_id = row.get("stop_id", "")
        try:
            stop_sequence = int(sequence_text)
        except Exception:
            stop_sequence = 10**9
        trips_rows[trip_id].append((stop_sequence, arrival_time, departure_time, stop_id))
    return trips_rows, total_rows


def group_trips_by_signature(trips_rows):
    """Group trip_ids that share an identical ordered stop-event signature.

    args:
        trips_rows: Mapping from trip_id to its stop-event tuples, from `build_trip_stop_events`.

    returns:
        Mapping from signature to the list of trip_ids sharing it.
    """
    signature_type = Tuple[Tuple[int, str, str, str], ...]
    signature_to_trip_ids: Dict[signature_type, List[str]] = defaultdict(list)
    max_workers = min(8, (os.cpu_count() or 4))
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(make_signature, item): item[0] for item in trips_rows.items()
        }
        for future in as_completed(futures):
            trip_id, signature = future.result()
            signature_to_trip_ids[signature].append(trip_id)
    return signature_to_trip_ids


def find_duplicate_groups(signature_to_trip_ids):
    """Return signature groups with 2+ trip_ids, sorted by group size then trip_ids.

    args:
        signature_to_trip_ids: Mapping from signature to trip_ids, from `group_trips_by_signature`.

    returns:
        List of (signature, sorted trip_ids) for groups with at least 2 trip_ids.
    """
    duplicate_groups = [
        (signature, sorted(trip_ids_for_signature))
        for signature, trip_ids_for_signature in signature_to_trip_ids.items()
        if len(trip_ids_for_signature) >= 2
    ]
    duplicate_groups.sort(key=lambda item: (len(item[1]), item[1]))
    return duplicate_groups


def print_duplicate_groups_summary(duplicate_groups):
    """Print a group-size histogram with one example group per size.

    args:
        duplicate_groups: Output of `find_duplicate_groups`.
    """
    total_duplicate_trip_ids = 0
    duplicate_group_sizes: Dict[int, int] = {}
    if not duplicate_groups:
        print("No pair/group of trip_id with identical sequence and schedules was found.")
        return

    total_duplicate_trip_ids = sum(len(trip_ids) for _, trip_ids in duplicate_groups)
    print(
        f"\nFound {len(duplicate_groups)} groups of trip_id with identical content "
        f"({total_duplicate_trip_ids} trip_id in total)."
    )

    for _, trip_ids_for_signature in duplicate_groups:
        group_size = len(trip_ids_for_signature)
        duplicate_group_sizes[group_size] = duplicate_group_sizes.get(group_size, 0) + 1
    for group_size in sorted(duplicate_group_sizes):
        print(f"Groups with {group_size} trip_id: {duplicate_group_sizes[group_size]}")
        for _, trip_ids_for_signature in duplicate_groups:
            if len(trip_ids_for_signature) == group_size:
                suffix = "..." if len(trip_ids_for_signature) > 5 else ""
                print(
                    f"  Example group with {group_size} trip_id:"
                    f" {trip_ids_for_signature[:5]}{suffix}"
                )
                break

In [ ]:
def main():
    """Find and export groups of trip_id with identical stop_times content."""
    trip_id_set = None
    trips_rows: Dict[str, List[Tuple[int, str, str, str]]] = {}
    total_rows = 0
    signature_to_trip_ids = None
    duplicate_groups = None
    total_duplicate_trip_ids = 0
    trip_ids_to_eliminate: List[str] = []
    eliminate_path = None
    check_missing_files([STOP_TIMES_SUBWAY_FILE, TRIPS_SUBWAY_FILE])

    print_file_disclaimer([
        (STOP_TIMES_SUBWAY_FILE, 'stop_times'),
        (TRIPS_SUBWAY_FILE, 'trips'),
    ])

    trip_id_set = load_trip_ids(TRIPS_SUBWAY_FILE)

    print(f"Unique trip_id in 'trips': {len(trip_id_set)}")

    trips_rows, total_rows = build_trip_stop_events(STOP_TIMES_SUBWAY_FILE, trip_id_set)

    print(f"Total rows read from 'stop_times': {total_rows}")
    print(f"trip_id with at least one row in 'stop_times': {len(trips_rows)}")

    signature_to_trip_ids = group_trips_by_signature(trips_rows)
    duplicate_groups = find_duplicate_groups(signature_to_trip_ids)

    print_duplicate_groups_summary(duplicate_groups)

    if not duplicate_groups:
        return

    total_duplicate_trip_ids = sum(
        len(trip_ids_for_signature) for _, trip_ids_for_signature in duplicate_groups
    )

    for _, trip_ids_for_signature in duplicate_groups:
        trip_ids_to_eliminate.extend(trip_ids_for_signature[1:])

    eliminate_path = os.path.join(DUPLICATED_TRIPS_BASE, "trip_ids_to_eliminate.txt")
    with open(eliminate_path, "w", encoding="utf-8") as file_handle:
        for trip_id in sorted(trip_ids_to_eliminate):
            file_handle.write(trip_id + "\n")

    print(
        f"\nTrip_id kept from groups (1 per group):"
        f" {total_duplicate_trip_ids - len(trip_ids_to_eliminate)}"
    )
    print(f"Trip_id to eliminate: {len(trip_ids_to_eliminate)}")
    print(
        f"Total valid trip_id after removing duplicates:"
        f" {len(trip_id_set) - len(trip_ids_to_eliminate)}"
    )
    print(
        f"{Path(TRIP_IDS_TO_ELIMINATE_FILE).name} generated in"
        f" {Path(TRIP_IDS_TO_ELIMINATE_FILE).relative_to(_project_root).parent}"
    )

main()

With `.src/gtfs/data/2_duplicated_trips/trip_ids_to_eliminate.txt` created in this cell, we create `trips_cleaned.txt` and `stop_times_cleaned.txt` in the same folder with `data_validation/processing/2_duplicated_trips.py`

##### Verifying it after deduplication

After running `data_validation/processing/2_duplicated_trips.py` we re-run the same check on `stop_times_cleaned.txt`/`trips_cleaned.txt`. Since exactly one representative trip_id was kept per duplicate-signature group, no signature should now map to 2+ trip_ids.

In [ ]:
def main():
    """Verify no duplicate trip_id blocs remain in stop_times after deduplication."""
    trip_id_set = None
    trips_rows: Dict[str, List[Tuple[int, str, str, str]]] = {}
    total_rows = 0
    signature_to_trip_ids = None
    duplicate_groups = None
    check_missing_files([STOP_TIMES_CLEANED_FILE, TRIPS_FILE])

    print_file_disclaimer([
        (STOP_TIMES_CLEANED_FILE, 'stop_times'),
        (TRIPS_FILE, 'trips'),
    ])

    trip_id_set = load_trip_ids(TRIPS_FILE)

    print(f"Unique trip_id in 'trips': {len(trip_id_set)}")

    trips_rows, total_rows = build_trip_stop_events(STOP_TIMES_CLEANED_FILE, trip_id_set)

    print(f"Total rows read from 'stop_times': {total_rows}")

    signature_to_trip_ids = group_trips_by_signature(trips_rows)
    duplicate_groups = find_duplicate_groups(signature_to_trip_ids)

    print_duplicate_groups_summary(duplicate_groups)

main()

#### The canonical stop sequence is followed correctly?

##### Verifying it for the original data

Mechanism: We use dictionaries from scripts/basics.py. Each route_name has a route_id in 'subway_routes_names_ids'. For each route_id, we gather trip_id values (and direction_id) from 'trips_subway_cleaned.txt'. For each trip_id, we read rows from 'stop_times_subway_cleaned.txt' ordered by stop_sequence and compare each consecutive pair with the expected order from 'subway_route_names_stop_ids' (reversed when direction_id=1). If a pair is flagged, we print only the two stops in bad order with their stop_sequence values and stop names from 'stops_subway_cleaned.txt'.

In [ ]:
def build_trip_to_route_dir(trips_file, rid_to_name):
    """Read trips and return a mapping of trip_id to (route_id, direction_id).

    args:
        trips_file: Path to the trips CSV file.
        rid_to_name: Mapping from route_id to route short name.

    returns:
        Tuple of the mapping dict and the count of matched rows.
    """
    trip_to_route_dir = {}
    matched_rows = 0
    for row in read_dict_rows(trips_file):
        trip_id = row.get("trip_id", "").strip()
        if not trip_id:
            continue
        route_id = row.get("route_id", "").strip()
        if route_id not in rid_to_name:
            continue
        matched_rows += 1
        trip_to_route_dir[trip_id] = (route_id, row.get("direction_id", "").strip() or "")
    return trip_to_route_dir, matched_rows


def build_trip_seq_rows(stop_times_file, trip_ids):
    """Read stop_times and return a mapping of trip_id to sorted (seq, stop_id) rows.

    args:
        stop_times_file: Path to the stop_times CSV file.
        trip_ids: Set of trip_id values to include.

    returns:
        Mapping from trip_id to its stops sorted by stop_sequence.
    """
    trip_seq_rows = defaultdict(list)
    for row in read_dict_rows(stop_times_file):
        trip_id = row.get("trip_id", "").strip()
        if trip_id not in trip_ids:
            continue
        stop_id = row.get("stop_id", "").strip()
        if not stop_id:
            continue
        try:
            stop_sequence = int(row.get("stop_sequence", "").strip())
        except Exception:
            continue
        trip_seq_rows[trip_id].append((stop_sequence, stop_id))
    for trip_id in trip_seq_rows:
        trip_seq_rows[trip_id].sort(key=lambda item: item[0])
    return trip_seq_rows


def detect_canonical_violations(
    trip_seq_rows, trip_to_route_dir, rid_to_name, route_names_stop_ids
):
    """Return trips whose consecutive stops break the canonical route order.

    Only stop_sequence pairs with a gap of exactly one are checked. Pairs
    separated by a larger gap are considered intentionally non-adjacent and skipped.

    args:
        trip_seq_rows: Mapping from trip_id to sorted (seq, stop_id) rows.
        trip_to_route_dir: Mapping from trip_id to (route_id, direction_id).
        rid_to_name: Mapping from route_id to route short name.
        route_names_stop_ids: Mapping from route name to canonical stop order.

    returns:
        List of (route_name, route_id, trip_id, direction_id, bad_pairs).
    """
    flagged = []
    for trip_id, seq_rows in trip_seq_rows.items():
        route_id, direction_id = trip_to_route_dir.get(trip_id, (None, None))
        if not route_id:
            continue
        route_name = rid_to_name.get(route_id)
        expected_order = list(route_names_stop_ids.get(route_name, []))
        if not expected_order:
            continue
        if direction_id == "1":
            expected_order = list(reversed(expected_order))
        adjacency = build_expected_adjacency(expected_order)
        bad_pairs = []
        for index in range(len(seq_rows) - 1):
            seq_a, stop_a = seq_rows[index]
            seq_b, stop_b = seq_rows[index + 1]
            if seq_b != seq_a + 1:
                continue
            if adjacency.get(stop_a) != stop_b:
                bad_pairs.append((seq_a, stop_a, seq_b, stop_b))
        if bad_pairs:
            flagged.append((route_name, route_id, trip_id, direction_id, bad_pairs))
    return flagged


def print_canonical_violations(flagged, stop_names):
    """Print each flagged trip with its out-of-order stop pairs.

    args:
        flagged: Output of detect_canonical_violations.
        stop_names: Mapping from stop_id to stop_name.
    """
    for route_name, route_id, trip_id, direction_id, bad_pairs in flagged:
        print(
            f"- route={route_name!r} route_id={route_id} trip_id={trip_id}"
            f" direction={direction_id} bad_pairs={len(bad_pairs)}"
        )
        for seq_a, stop_a, seq_b, stop_b in bad_pairs:
            print(
                f"    bad order: [{seq_a}] {stop_a} ({stop_names.get(stop_a, '(no name)')}) -> "
                f"[{seq_b}] {stop_b} ({stop_names.get(stop_b, '(no name)')})"
            )

In [ ]:
def main():
    """Validate that subway trips follow the canonical stop sequence for their route."""
    rid_to_name = None
    trip_to_route_dir = None
    matched_rows = 0
    available_route_ids = None
    suffix = None
    trip_ids = None
    trip_seq_rows = None
    stop_names = None
    flagged = None
    total_bad_pairs = 0
    check_missing_files([TRIPS_FILE, STOP_TIMES_CLEANED_FILE, STOPS_FILE])

    print_file_disclaimer([
        (STOP_TIMES_CLEANED_FILE, 'stop_times'),
        (STOPS_FILE, 'stops'),
        (TRIPS_FILE, 'trips'),
    ])

    rid_to_name = {rid: name for name, rid in basics.subway_routes_names_ids.items()}
    trip_to_route_dir, matched_rows = build_trip_to_route_dir(TRIPS_FILE, rid_to_name)

    if not trip_to_route_dir:
        available_route_ids = sorted(
            {
                row.get("route_id", "").strip()
                for row in read_dict_rows(TRIPS_FILE)
                if row.get("route_id", "").strip()
            }
        )
        suffix = "" if len(available_route_ids) <= 20 else " ..."
        print("No subway trips found in 'trips' for the canonical mappings.")
        print(f"Trips rows matching subway route_ids: {matched_rows}")
        print(f"Available route_id values in 'trips': {available_route_ids[:20]}{suffix}")
        return

    trip_ids = set(trip_to_route_dir)
    trip_seq_rows = build_trip_seq_rows(STOP_TIMES_CLEANED_FILE, trip_ids)
    stop_names = load_stop_names(STOPS_FILE)
    flagged = detect_canonical_violations(
        trip_seq_rows, trip_to_route_dir, rid_to_name, basics.subway_route_names_stop_ids
    )

    os.makedirs(STOP_SEQUENCE_BASE, exist_ok=True)
    with open(WRONG_STOP_SEQUENCES_FILE, "w", encoding="utf-8", newline="") as fh:
        writer = csv.DictWriter(
            fh, fieldnames=["trip_id", "stop_a", "stop_b", "seq_a", "seq_b"]
        )
        writer.writeheader()
        for _, _, trip_id, _, bad_pairs in flagged:
            for seq_a, stop_a, seq_b, stop_b in bad_pairs:
                writer.writerow({
                    "trip_id": trip_id, "stop_a": stop_a, "stop_b": stop_b,
                    "seq_a": seq_a, "seq_b": seq_b,
                })
                total_bad_pairs += 1

    if not flagged:
        print("All checked subway trips follow an allowed contiguous stop sequence.")
        print(
            f"{Path(WRONG_STOP_SEQUENCES_FILE).name} (empty) written"
            f" to {Path(WRONG_STOP_SEQUENCES_FILE).relative_to(_project_root)}"
        )
        return

    print(f"FOUND {len(flagged)} trips with unexpected stop sequences:")
    print_canonical_violations(flagged, stop_names)
    print(
        f"\n{Path(WRONG_STOP_SEQUENCES_FILE).name} written"
        f" to {Path(WRONG_STOP_SEQUENCES_FILE).relative_to(_project_root)}"
        f" ({total_bad_pairs} rows)"
    )

main()

##### Verifying it after the sequence fix



After running `data_validation/processing/3_stop_sequence.py` we re-run the same check on `stop_times_sequence.txt`. The same trips will still appear as having non-canonical order (the stops themselves have not changed), but the bad pairs should now show non-consecutive stop_sequence numbers, confirming that the gap was correctly inserted.

In [ ]:
def main():
    """Verify canonical stop order in stop_times_sequence after the sequence fix."""
    rid_to_name = None
    trip_to_route_dir = None
    matched_rows = 0
    available_route_ids = None
    suffix = None
    trip_ids = None
    trip_seq_rows = None
    stop_names = None
    flagged = None
    check_missing_files([TRIPS_FILE, STOP_TIMES_SEQUENCE_FILE, STOPS_FILE])

    print_file_disclaimer([
        (STOP_TIMES_SEQUENCE_FILE, 'stop_times'),
        (STOPS_FILE, 'stops'),
        (TRIPS_FILE, 'trips'),
    ])

    rid_to_name = {rid: name for name, rid in basics.subway_routes_names_ids.items()}
    trip_to_route_dir, matched_rows = build_trip_to_route_dir(TRIPS_FILE, rid_to_name)

    if not trip_to_route_dir:
        available_route_ids = sorted(
            {
                row.get("route_id", "").strip()
                for row in read_dict_rows(TRIPS_FILE)
                if row.get("route_id", "").strip()
            }
        )
        suffix = "" if len(available_route_ids) <= 20 else " ..."
        print("No subway trips found in 'trips' for the canonical mappings.")
        print(f"Trips rows matching subway route_ids: {matched_rows}")
        print(f"Available route_id values in 'trips': {available_route_ids[:20]}{suffix}")
        return

    trip_ids = set(trip_to_route_dir)
    trip_seq_rows = build_trip_seq_rows(STOP_TIMES_SEQUENCE_FILE, trip_ids)
    stop_names = load_stop_names(STOPS_FILE)
    flagged = detect_canonical_violations(
        trip_seq_rows, trip_to_route_dir, rid_to_name, basics.subway_route_names_stop_ids
    )

    if not flagged:
        print("All checked subway trips follow an allowed contiguous stop sequence.")
        return

    print(f"FOUND {len(flagged)} trips with unexpected stop sequences:")
    print_canonical_violations(flagged, stop_names)

main()

#### Cases when arrival_time and deparature_time is the same

##### Which stops have the same arrival_time and deparature_time?

Goal: identify which stop_ids have arrival_time == departure_time in 'stop_times', broken down by subway line, and determine what fraction of those occurrences happen at terminal stops (first or last stop of the trip). For stops where only some trips have arrival == departure, also compute the mean and standard deviation of the actual door time (departure - arrival) across the trips where they differ.

In [ ]:
def build_arr_dep_counts(stop_times_file, trips_file, rid_to_name):
    """Build (line, stop_id) -> [equal_count, terminal_count, total_count, sample_trip_ids, door_times].

    args:
        stop_times_file: Path to the stop_times file to scan.
        trips_file: Path to the trips file used to resolve trip_id -> line.
        rid_to_name: Mapping from route_id to line name.

    returns:
        Mapping from (line, stop_id) to its arrival/departure counters.
    """
    trip_to_line = load_trip_to_line(trips_file, rid_to_name)
    trip_bounds = load_trip_sequence_bounds(stop_times_file, set(trip_to_line))

    counts = defaultdict(lambda: [0, 0, 0, [], []])
    for row in read_dict_rows(stop_times_file):
        trip_id = row.get("trip_id", "").strip()
        line = trip_to_line.get(trip_id)
        if not line:
            continue
        stop_id = row.get("stop_id", "").strip()
        if not stop_id:
            continue
        arrival = row.get("arrival_time", "").strip()
        departure = row.get("departure_time", "").strip()
        if not arrival or not departure:
            continue
        key = (line, stop_id)
        counts[key][2] += 1
        if arrival == departure:
            counts[key][0] += 1
            try:
                seq = int(row.get("stop_sequence", "").strip())
                min_seq, max_seq = trip_bounds.get(trip_id, (None, None))
                if (
                    min_seq is not None and max_seq is not None
                    and (seq == min_seq or seq == max_seq)
                ):
                    counts[key][1] += 1
            except Exception:
                pass
            if len(counts[key][3]) < 5:
                counts[key][3].append(trip_id)
        else:
            try:
                door_time = parse_time_to_seconds(departure) - parse_time_to_seconds(arrival)
                if door_time >= 0:
                    counts[key][4].append(door_time)
            except Exception:
                pass
    return counts


def print_arr_dep_table(counts, stop_names):
    """Print the per-(line, stop_id) arrival=departure summary table.

    Only stops with at least one arrival==departure occurrence are printed.

    args:
        counts: Mapping from build_arr_dep_counts.
        stop_names: Mapping from stop_id to stop_name.
    """
    col_line = 6
    col_stop_id = 12
    col_stop_name = 40
    col_equal = 18
    col_terminal = 10
    col_total = 8
    header = (
        f"{'line':<{col_line}} {'stop_id':<{col_stop_id}}"
        f" {'stop_name':<{col_stop_name}} {'arrival=departure':<{col_equal}}"
        f" {'terminal%':<{col_terminal}} {'total':<{col_total}} sample_trip_ids"
    )
    print(header)
    print("-" * len(header))

    for line in basics.subway_routes_names_ids:
        for stop_id in basics.subway_route_names_stop_ids.get(line, []):
            key = (line, stop_id)
            if key not in counts:
                continue
            equal_count, terminal_count, total_count, sample_trips, _ = counts[key]
            if equal_count == 0:
                continue
            stop_name = stop_names.get(stop_id, "(no name)")
            terminal_pct = f"{terminal_count / equal_count:.0%}"
            print(
                f"{line:<{col_line}} {stop_id:<{col_stop_id}}"
                f" {stop_name:<{col_stop_name}} {equal_count:<{col_equal}}"
                f" {terminal_pct:<{col_terminal}} {total_count:<{col_total}}"
                f" {', '.join(sample_trips)}"
            )

In [ ]:
def main():
    """Build arr=dep counts per (line, stop_id) and print the summary table.

    Returns counts and stop_names for use in the door-times cell below.
    """
    rid_to_name = None
    counts = None
    stop_names = None
    equal_stop_ids = None
    check_missing_files([STOP_TIMES_SEQUENCE_FILE, TRIPS_FILE, STOPS_FILE])

    print_file_disclaimer([
        (STOP_TIMES_SEQUENCE_FILE, 'stop_times'),
        (TRIPS_FILE, 'trips'),
        (STOPS_FILE, 'stops'),
    ])

    rid_to_name = {rid: name for name, rid in basics.subway_routes_names_ids.items()}
    counts = build_arr_dep_counts(STOP_TIMES_SEQUENCE_FILE, TRIPS_FILE, rid_to_name)

    equal_stop_ids = {sid for (_, sid), (eq, *_) in counts.items() if eq > 0}
    stop_names = load_stop_names(STOPS_FILE, stop_ids=equal_stop_ids)

    print_arr_dep_table(counts, stop_names)

    return counts, stop_names

_arr_dep_counts, _arr_dep_stop_names = main()

##### For the non-canonical terminal stops, what is its usual doors-time?

Goal: for stops where only some trips have arrival == departure (i.e. not canonical terminals), print the mean and standard deviation of the actual door time (departure - arrival in seconds) using only the trips where they differ. The counts and stop names built above are reused directly.

In [ ]:
def main(counts, stop_names):
    """Print door times for stops with partial arrival=departure (excluding arr==dep rows).

    args:
        counts: Mapping from (line, stop_id) to [equal_count, _, total_count, _, door_times].
        stop_names: Mapping from stop_id to stop_name.
    """
    col_line = 6
    col_stop_id = 12
    col_stop_name = 40
    col_mean = 10
    col_std = 10
    header = (
        f"{'line':<{col_line}} {'stop_id':<{col_stop_id}}"
        f" {'stop_name':<{col_stop_name}} {'mean(s)':<{col_mean}} {'stdev(s)':<{col_std}} n"
    )
    print(header)
    print("-" * len(header))

    for line in basics.subway_routes_names_ids:
        for stop_id in basics.subway_route_names_stop_ids.get(line, []):
            key = (line, stop_id)
            if key not in counts:
                continue
            equal_count, _, total_count, _, door_times = counts[key]
            if not (0 < equal_count < total_count and door_times):
                continue
            stop_name = stop_names.get(stop_id, "(no name)")
            m = mean(door_times)
            s = stdev(door_times) if len(door_times) > 1 else 0.0
            print(
                f"{line:<{col_line}} {stop_id:<{col_stop_id}}"
                f" {stop_name:<{col_stop_name}}"
                f" {m:<{col_mean}.2f} {s:<{col_std}.2f} {len(door_times)}"
            )

main(_arr_dep_counts, _arr_dep_stop_names)

##### Door-time per line

Goal: compute a single door-time estimate per subway line - the mean and standard deviation of departure - arrival across all stops and trips, excluding rows where arrival == departure.

In [ ]:
def main():
    """Compute mean and stdev of door time (dep - arr) per line, excluding arr == dep rows."""
    rid_to_name = None
    trip_to_line = None
    door_times_by_line = None
    col_line = 0
    col_mean = 0
    col_std = 0
    header = None
    check_missing_files([STOP_TIMES_SEQUENCE_FILE, TRIPS_FILE])

    print_file_disclaimer([
        (STOP_TIMES_SEQUENCE_FILE, 'stop_times'),
        (TRIPS_FILE, 'trips'),
    ])

    rid_to_name = {rid: name for name, rid in basics.subway_routes_names_ids.items()}
    trip_to_line = load_trip_to_line(TRIPS_FILE, rid_to_name)

    door_times_by_line = defaultdict(list)
    for row in read_dict_rows(STOP_TIMES_SEQUENCE_FILE):
        trip_id = row.get("trip_id", "").strip()
        line = trip_to_line.get(trip_id)
        if not line:
            continue
        arrival = row.get("arrival_time", "").strip()
        departure = row.get("departure_time", "").strip()
        if not arrival or not departure or arrival == departure:
            continue
        try:
            door_time = parse_time_to_seconds(departure) - parse_time_to_seconds(arrival)
            if door_time >= 0:
                door_times_by_line[line].append(door_time)
        except Exception:
            pass

    col_line = 6
    col_mean = 10
    col_std = 10
    header = f"{'line':<{col_line}} {'mean(s)':<{col_mean}} {'stdev(s)':<{col_std}} n"
    print(header)
    print("-" * len(header))

    for line in basics.subway_routes_names_ids:
        door_times = door_times_by_line.get(line, [])
        if not door_times:
            print(f"{line:<{col_line}} {'N/A':<{col_mean}} {'N/A':<{col_std}} 0")
            continue
        m = mean(door_times)
        s = stdev(door_times) if len(door_times) > 1 else 0.0
        print(f"{line:<{col_line}} {m:<{col_mean}.2f} {s:<{col_std}.2f} {len(door_times)}")

main()

##### Write doors.txt

Goal: produce `doors.txt` in `.src/gtfs/data/4_doors_time/` with one row per (stop_id, line) that has any arrival == departure occurrence. Canonical terminals (all trips have arr == dep) receive their line mean door time; partial stops receive their per-stop mean. FM has no observed door times, so it uses the mean across all other lines as a fallback.

In [ ]:
def main():
    """Write doors.txt: (stop_id, line, door_seconds) for every stop with arr==dep."""
    door_times_by_line = None
    all_other_times = None
    fm_fallback = 0
    rows = None

    # Aggregate integer door times per line
    door_times_by_line = defaultdict(list)
    for (line, _), (_, _, _, _, door_times) in _arr_dep_counts.items():
        door_times_by_line[line].extend(door_times)

    # FM fallback: mean of all other lines with data
    all_other_times = [
        t for line, times in door_times_by_line.items() if line != "FM" for t in times
    ]
    fm_fallback = round_half_up_mean(all_other_times)

    rows = []
    for line in basics.subway_routes_names_ids:
        for stop_id in basics.subway_route_names_stop_ids.get(line, []):
            key = (line, stop_id)
            if key not in _arr_dep_counts:
                continue
            equal_count, _, total_count, _, door_times = _arr_dep_counts[key]
            if equal_count == 0:
                continue
            if equal_count == total_count:
                line_times = door_times_by_line.get(line, [])
                door_seconds = (
                    fm_fallback if not line_times else round_half_up_mean(line_times)
                )
            else:
                door_seconds = round_half_up_mean(door_times)
            rows.append({"stop_id": stop_id, "line": line, "door_seconds": door_seconds})

    os.makedirs(DOORS_BASE, exist_ok=True)
    with open(DOORS_FILE, "w", encoding="utf-8", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=["stop_id", "line", "door_seconds"])
        writer.writeheader()
        writer.writerows(rows)
    print(
        f"doors.txt written to {Path(DOORS_FILE).relative_to(_project_root)}"
        f" ({len(rows)} rows)\n"
    )

main()

##### Verifying it after the door-time fix

After running `data_validation/processing/4_doors_time.py` we re-run the same arrival/departure check on `stop_times_doors.txt`. Any stop that still has `arrival_time == departure_time` would mean the door-time fix missed it.

In [ ]:
def main():
    """Verify no stop has arrival_time == departure_time after applying door times."""
    rid_to_name = None
    counts = None
    stop_names = None
    equal_stop_ids = None
    check_missing_files([STOP_TIMES_FILE, TRIPS_FILE, STOPS_FILE])

    print_file_disclaimer([
        (STOP_TIMES_FILE, 'stop_times'),
        (TRIPS_FILE, 'trips'),
        (STOPS_FILE, 'stops'),
    ])

    rid_to_name = {rid: name for name, rid in basics.subway_routes_names_ids.items()}
    counts = build_arr_dep_counts(STOP_TIMES_FILE, TRIPS_FILE, rid_to_name)

    equal_stop_ids = {sid for (_, sid), (eq, *_) in counts.items() if eq > 0}

    if not equal_stop_ids:
        print(
            "All correct: no stop has arrival_time == departure_time"
            " after applying door times."
        )
        return

    stop_names = load_stop_names(STOPS_FILE, stop_ids=equal_stop_ids)
    print(f"FOUND {len(equal_stop_ids)} stops still with arrival_time == departure_time:")
    print_arr_dep_table(counts, stop_names)

main()